In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Conv1D, MaxPooling1D, AveragePooling1D, Flatten, Dense, Reshape, LSTM
from tensorflow.keras.layers import Add, Concatenate, Input, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from imblearn.over_sampling import SMOTE

def load_and_preprocess_data(file_path):
    df = pd.read_csv(file_path)
    label_encoders = {}

    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
    for col in categorical_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        label_encoders[col] = le

    X = df.drop(columns=["churn"])
    y = df["churn"].values

    #split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # perform feature selection
    rfe = RFE(estimator=RandomForestClassifier(), n_features_to_select=10)
    X_train_rfe = rfe.fit_transform(X_train, y_train)
    X_test_rfe = rfe.transform(X_test)#, y_test)

    # balance classes
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_rfe, y_train)

    #---> Apply StandardScaler to X
    X_standard_scaler = StandardScaler()
    X_train_std = X_standard_scaler.fit_transform(X_train_resampled)
    X_test_std = X_standard_scaler.transform(X_test_rfe)

    # Apply MinMaxScaler after StandardScaler
    # ------> MinMaxScaler IS ALWAYS APPLIED AFTER
    # ------> StandardScaler
    X_min_max_scaler = MinMaxScaler()
    X_train_scaled = X_min_max_scaler.fit_transform(X_train_std)
    X_test_scaled = X_min_max_scaler.transform(X_test_std)

    # Reshape data
    X_train_scaled = X_train_scaled.reshape((X_train_scaled.shape[0], X_train_scaled.shape[1], 1))
    X_test_scaled = X_test_scaled.reshape((X_test_scaled.shape[0], X_test_scaled.shape[1], 1))

    return X_train_scaled, y_train_resampled, X_test_scaled, y_test, X_train_scaled.shape, X_test_scaled.shape, X_standard_scaler, X_min_max_scaler


def build_model(input_shape):
    inputs = Input(shape=input_shape)

    # First Conv1D block
    conv1_block1 = Conv1D(filters=16, kernel_size=3, activation='relu', padding='same')(inputs)
    conv2_block1 = Conv1D(filters=16, kernel_size=5, activation='relu', padding='same')(inputs)
    conv3_block1 = Conv1D(filters=16, kernel_size=7, activation='relu', padding='same')(inputs)

    concat_block1 = Concatenate()([conv1_block1, conv2_block1, conv3_block1])
    pool_block1 =AveragePooling1D(pool_size=3, padding='same')(concat_block1)
    dropout_block1 = Dropout(0.2)(pool_block1)

    # Second Conv1D block
    conv1_block2 = Conv1D(filters=32, kernel_size=3, activation='relu', padding='same')(dropout_block1)
    conv2_block2 = Conv1D(filters=32, kernel_size=5, activation='relu', padding='same')(dropout_block1)
    conv3_block2 = Conv1D(filters=32, kernel_size=7, activation='relu', padding='same')(dropout_block1)

    concat_block2 = Concatenate()([conv1_block2, conv2_block2, conv3_block2])
    pool_block2 =AveragePooling1D(pool_size=3, padding='same')(concat_block2)
    dropout_block2 = Dropout(0.5)(pool_block2)

    # Flatten and reshape before LSTM
    #flat = Flatten()(dropout_block2)
    #reshaped = Reshape((1, -1))(flat)
    lstm = LSTM(128, return_sequences=False)(dropout_block2)

    # Fully connected output
    fc = Dense(128, activation='relu')(lstm)
    output = Dense(1, activation='sigmoid')(fc)

    model = Model(inputs=inputs, outputs=output)
    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    model.summary()
    return model



def train_model(model, X_train, y_train):
    early_stopping = EarlyStopping(monitor='val_loss', patience=50, restore_best_weights=True, verbose=1)

    history = model.fit(
        X_train, y_train,
        epochs=300,
        batch_size=64,
        validation_split=0.20,
        callbacks=[early_stopping],
        verbose=1
    )

    return history

def evaluate_model(model, X_test, y_test):

    test_loss, test_acc = model.evaluate(X_test, y_test)
    return test_loss, test_acc

def run_churn_model():
    file_path_train = "/content/drive/MyDrive/telecom_churn.csv"

    X_train, y_train, X_test, y_test, X_train_shape, X_test_shape, X_std_scaler, X_mm_scaler=load_and_preprocess_data(file_path_train)

    # Print shapes
    print("\nDataset Shapes:")
    print(f"Training set: {X_train_shape}")
    print(f"Test set: {X_test_shape}\n")

    # Build the model
    model = build_model((X_train.shape[1], 1))

    # Train the model
    history = train_model(model, X_train, y_train)

    # Track the number of epochs where early stopping occurred
    early_stopped_epoch = len(history.history['loss'])

    # ---> EVALUATE MODEL ON X_test_rfe
    test_loss, test_acc = evaluate_model(model, X_test, y_test)

    # Print results
    final_train_accuracy = history.history['accuracy'][-1]
    print(f"\nFinal Training Accuracy: {final_train_accuracy:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")

run_churn_model()



Dataset Shapes:
Training set: (4568, 10, 1)
Test set: (667, 10, 1)



Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)  │ (None, 10, 1)          │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d (Conv1D)           │ (None, 10, 16)         │             64 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_1 (Conv1D)         │ (None, 10, 16)         │             96 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_2 (Conv1D)         │ (None, 10, 16)         │            128 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate (Concatenate) │ (None, 10, 48)         │              0 │ conv1d[0][0],          │
│                           │                        │                │ conv1d_1[0][0],        │
│                           │                        │                │ conv1d_2[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ average_pooling1d         │ (None, 4, 48)          │              0 │ concatenate[0][0]      │
│ (AveragePooling1D)        │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout (Dropout)         │ (None, 4, 48)          │              0 │ average_pooling1d[0][… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_3 (Conv1D)         │ (None, 4, 32)          │          4,640 │ dropout[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_4 (Conv1D)         │ (None, 4, 32)          │          7,712 │ dropout[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_5 (Conv1D)         │ (None, 4, 32)          │         10,784 │ dropout[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate_1             │ (None, 4, 96)          │              0 │ conv1d_3[0][0],        │
│ (Concatenate)             │                        │                │ conv1d_4[0][0],        │
│                           │                        │                │ conv1d_5[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ average_pooling1d_1       │ (None, 2, 96)          │              0 │ concatenate_1[0][0]    │
│ (AveragePooling1D)        │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_1 (Dropout)       │ (None, 2, 96)          │              0 │ average_pooling1d_1[0… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ lstm (LSTM)               │ (None, 128)            │        115,200 │ dropout_1[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense (Dense)             │ (None, 128)            │         16,512 │ lstm[0][0]             │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_1 (Dense)           │ (None, 1)              │            129 │ dense[0][0]            │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 155,265 (606.50 KB)

 Trainable params: 155,265 (606.50 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/300
58/58 ━━━━━━━━━━━━━━━━━━━━ 15s 34ms/step - accuracy: 0.6094 - loss: 0.6729 - val_accuracy: 0.1532 - val_loss: 0.8205
Epoch 2/300
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6776 - loss: 0.6191 - val_accuracy: 0.6258 - val_loss: 0.6457
Epoch 3/300
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7131 - loss: 0.5669 - val_accuracy: 0.3348 - val_loss: 0.9869
Epoch 4/300
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7252 - loss: 0.5474 - val_accuracy: 0.7801 - val_loss: 0.5751
Epoch 5/300
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7418 - loss: 0.5259 - val_accuracy: 0.5131 - val_loss: 0.8954
Epoch 6/300
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7491 - loss: 0.5073 - val_accuracy: 0.5799 - val_loss: 0.8148
Epoch 7/300
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7697 - loss: 0.4881 - val_accuracy: 0.6444 - val_loss: 0.7292
Epoch 8/300
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.7843 - loss: 0.4691 - val_accuracy: 0.

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Concatenate, Input, Dropout, Reshape, LSTM
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from imblearn.over_sampling import SMOTE

def load_and_preprocess_data(file_path):
    df = pd.read_csv(file_path)
    label_encoders = {}

    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
    for col in categorical_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        label_encoders[col] = le

    X = df.drop(columns=["churn"])
    y = df["churn"].values

    #split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # perform feature selection
    rfe = RFE(estimator=RandomForestClassifier(), n_features_to_select=10)
    X_train_rfe = rfe.fit_transform(X_train, y_train)
    X_test_rfe = rfe.transform(X_test)#, y_test)

    # balance classes
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_rfe, y_train)

    #---> Apply StandardScaler to X
    X_standard_scaler = StandardScaler()
    X_train_std = X_standard_scaler.fit_transform(X_train_resampled)
    X_test_std = X_standard_scaler.transform(X_test_rfe)

    # Apply MinMaxScaler after StandardScaler
    # ------> MinMaxScaler IS ALWAYS APPLIED AFTER
    # ------> StandardScaler
    X_min_max_scaler = MinMaxScaler()
    X_train_scaled = X_min_max_scaler.fit_transform(X_train_std)
    X_test_scaled = X_min_max_scaler.transform(X_test_std)

    # Reshape data
    X_train_scaled = X_train_scaled.reshape((X_train_scaled.shape[0], X_train_scaled.shape[1], 1))
    X_test_scaled = X_test_scaled.reshape((X_test_scaled.shape[0], X_test_scaled.shape[1], 1))

    return X_train_scaled, y_train_resampled, X_test_scaled, y_test, X_train_scaled.shape, X_test_scaled.shape, X_standard_scaler, X_min_max_scaler



def build_model(input_shape):
    inputs = Input(shape=input_shape)

    # First Conv1D block
    conv1_block1 = Conv1D(filters=16, kernel_size=3, activation='relu', padding='same')(inputs)
    conv2_block1 = Conv1D(filters=16, kernel_size=5, activation='relu', padding='same')(inputs)
    conv3_block1 = Conv1D(filters=16, kernel_size=7, activation='relu', padding='same')(inputs)

    concat_block1 = Concatenate()([conv1_block1, conv2_block1, conv3_block1])
    pool_block1 = AveragePooling1D(pool_size=3, padding='same')(concat_block1)
    dropout_block1 = Dropout(0.2)(pool_block1)

    # Second Conv1D block
    conv1_block2 = Conv1D(filters=32, kernel_size=3, activation='relu', padding='same')(dropout_block1)
    conv2_block2 = Conv1D(filters=32, kernel_size=5, activation='relu', padding='same')(dropout_block1)
    conv3_block2 = Conv1D(filters=32, kernel_size=7, activation='relu', padding='same')(dropout_block1)

    concat_block2 = Concatenate()([conv1_block2, conv2_block2, conv3_block2])
    pool_block2 = AveragePooling1D(pool_size=3, padding='same')(concat_block2)
    dropout_block2 = Dropout(0.5)(pool_block2)

    # Flatten and reshape before LSTM
    #flat = Flatten()(dropout_block2)
    #reshaped = Reshape((1, -1))(flat)
    lstm = LSTM(128, return_sequences=False)(dropout_block2)

    # Fully connected output
    fc = Dense(128, activation='relu')(lstm)
    output = Dense(1, activation='sigmoid')(fc)

    model = Model(inputs=inputs, outputs=output)
    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    model.summary()
    return model


def train_model(model, X_train, y_train):

    history = model.fit(
        X_train, y_train,
        epochs=99,
        batch_size=64,
        verbose=1
    )

    return history

def evaluate_model(model, X_test, y_test):

    test_loss, test_acc = model.evaluate(X_test, y_test)
    return test_loss, test_acc

def run_churn_model():
    file_path_train = "/content/drive/MyDrive/telecom_churn.csv"

    X_train, y_train, X_test, y_test, X_train_shape, X_test_shape, X_standard_scaler, X_min_max_scaler=load_and_preprocess_data(file_path_train)

    # Print shapes
    print("\nDataset Shapes:")
    print(f"Training set: {X_train_shape}")
    print(f"Test set: {X_test_shape}\n")

    # Build the model
    model = build_model((X_train.shape[1], 1))

    # Train the model
    history = train_model(model, X_train, y_train)

    # Track the number of epochs where early stopping occurred
    early_stopped_epoch = len(history.history['loss'])

    # ---> EVALUATE MODEL ON X_test_rfe
    test_loss, test_acc = evaluate_model(model, X_test, y_test)

    # Print results
    final_train_accuracy = history.history['accuracy'][-1]
    print(f"\nFinal Training Accuracy: {final_train_accuracy:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")

run_churn_model()



Dataset Shapes:
Training set: (4568, 10, 1)
Test set: (667, 10, 1)



Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1             │ (None, 10, 1)          │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_6 (Conv1D)         │ (None, 10, 16)         │             64 │ input_layer_1[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_7 (Conv1D)         │ (None, 10, 16)         │             96 │ input_layer_1[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_8 (Conv1D)         │ (None, 10, 16)         │            128 │ input_layer_1[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate_2             │ (None, 10, 48)         │              0 │ conv1d_6[0][0],        │
│ (Concatenate)             │                        │                │ conv1d_7[0][0],        │
│                           │                        │                │ conv1d_8[0][0]         │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ average_pooling1d_2       │ (None, 4, 48)          │              0 │ concatenate_2[0][0]    │
│ (AveragePooling1D)        │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_2 (Dropout)       │ (None, 4, 48)          │              0 │ average_pooling1d_2[0… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_9 (Conv1D)         │ (None, 4, 32)          │          4,640 │ dropout_2[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_10 (Conv1D)        │ (None, 4, 32)          │          7,712 │ dropout_2[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_11 (Conv1D)        │ (None, 4, 32)          │         10,784 │ dropout_2[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate_3             │ (None, 4, 96)          │              0 │ conv1d_9[0][0],        │
│ (Concatenate)             │                        │                │ conv1d_10[0][0],       │
│                           │                        │                │ conv1d_11[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ average_pooling1d_3       │ (None, 2, 96)          │              0 │ concatenate_3[0][0]    │
│ (AveragePooling1D)        │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_3 (Dropout)       │ (None, 2, 96)          │              0 │ average_pooling1d_3[0… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ lstm_1 (LSTM)             │ (None, 128)            │        115,200 │ dropout_3[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_2 (Dense)           │ (None, 128)            │         16,512 │ lstm_1[0][0]           │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_3 (Dense)           │ (None, 1)              │            129 │ dense_2[0][0]          │
└──────────────────────

 Total params: 155,265 (606.50 KB)

 Trainable params: 155,265 (606.50 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/99
72/72 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.5506 - loss: 0.6875
Epoch 2/99
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.6755 - loss: 0.6123
Epoch 3/99
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7489 - loss: 0.5320
Epoch 4/99
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.7754 - loss: 0.4967
Epoch 5/99
72/72 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.7736 - loss: 0.5050
Epoch 6/99
72/72 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.7940 - loss: 0.4807
Epoch 7/99
72/72 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.8083 - loss: 0.4654
Epoch 8/99
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.8185 - loss: 0.4377
Epoch 9/99
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8216 - loss: 0.4439
Epoch 10/99
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.8317 - loss: 0.4237
Epoch 11/99
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8372 - loss: 0.4172
Epoch 12/99
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Conv1D, MaxPooling1D, AveragePooling1D, Flatten, Dense, Reshape, LSTM
from tensorflow.keras.layers import Add, Concatenate, Input, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from imblearn.over_sampling import SMOTE

def load_and_preprocess_data(file_path):
    df = pd.read_csv(file_path)
    label_encoders = {}

    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
    for col in categorical_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        label_encoders[col] = le

    X = df.drop(columns=["churn"])
    y = df["churn"].values

    #split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # perform feature selection
    rfe = RFE(estimator=RandomForestClassifier(), n_features_to_select=15)
    X_train_rfe = rfe.fit_transform(X_train, y_train)
    X_test_rfe = rfe.transform(X_test)#, y_test)

    # balance classes
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_rfe, y_train)

    #---> Apply StandardScaler to X
    X_standard_scaler = StandardScaler()
    X_train_std = X_standard_scaler.fit_transform(X_train_resampled)
    X_test_std = X_standard_scaler.transform(X_test_rfe)

    # Apply MinMaxScaler after StandardScaler
    # ------> MinMaxScaler IS ALWAYS APPLIED AFTER
    # ------> StandardScaler
    X_min_max_scaler = MinMaxScaler()
    X_train_scaled = X_min_max_scaler.fit_transform(X_train_std)
    X_test_scaled = X_min_max_scaler.transform(X_test_std)

    # Reshape data
    X_train_scaled = X_train_scaled.reshape((X_train_scaled.shape[0], X_train_scaled.shape[1], 1))
    X_test_scaled = X_test_scaled.reshape((X_test_scaled.shape[0], X_test_scaled.shape[1], 1))

    return X_train_scaled, y_train_resampled, X_test_scaled, y_test, X_train_scaled.shape, X_test_scaled.shape, X_standard_scaler, X_min_max_scaler

def build_model(input_shape):
    inputs = Input(shape=input_shape)

    # First Conv1D block
    conv1_block1 = Conv1D(filters=16, kernel_size=3, activation='relu', padding='same')(inputs)
    conv2_block1 = Conv1D(filters=16, kernel_size=5, activation='relu', padding='same')(inputs)
    conv3_block1 = Conv1D(filters=16, kernel_size=7, activation='relu', padding='same')(inputs)

    concat_block1 = Concatenate()([conv1_block1, conv2_block1, conv3_block1])
    projection1 = Conv1D(filters=concat_block1.shape[-1], kernel_size=1, padding='same')(inputs)
    skip1 = Add()([concat_block1, projection1])
    pool_block1 = AveragePooling1D(pool_size=3, padding='same')(skip1)
    dropout_block1 = Dropout(0.2)(pool_block1)

    # Second Conv1D block
    conv1_block2 = Conv1D(filters=32, kernel_size=3, activation='relu', padding='same')(dropout_block1)
    conv2_block2 = Conv1D(filters=32, kernel_size=5, activation='relu', padding='same')(dropout_block1)
    conv3_block2 = Conv1D(filters=32, kernel_size=7, activation='relu', padding='same')(dropout_block1)

    concat_block2 = Concatenate()([conv1_block2, conv2_block2, conv3_block2])
    projection2 = Conv1D(filters=concat_block2.shape[-1], kernel_size=1, padding='same')(dropout_block1)
    skip2 = Add()([concat_block2, projection2])
    pool_block2 =AveragePooling1D(pool_size=3, padding='same')(skip2)
    dropout_block2 = Dropout(0.25)(pool_block2)

    # Flatten and reshape before LSTM
    flat = Flatten()(dropout_block2)
    reshaped = Reshape((1, -1))(flat)
    lstm = LSTM(64, return_sequences=False)(reshaped)

    # Fully connected output
    fc = Dense(128, activation='relu')(lstm)
    output = Dense(1, activation='sigmoid')(fc)

    model = Model(inputs=inputs, outputs=output)
    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    model.summary()
    return model



def train_model(model, X_train, y_train):
    early_stopping = EarlyStopping(monitor='val_loss', patience=50, restore_best_weights=True, verbose=1)

    history = model.fit(
        X_train, y_train,
        epochs=300,
        batch_size=64,
        validation_split=0.30,
        callbacks=[early_stopping],
        verbose=1
    )

    return history

def evaluate_model(model, X_test, y_test):

    test_loss, test_acc = model.evaluate(X_test, y_test)
    return test_loss, test_acc

def run_churn_model():
    file_path_train = "/content/drive/MyDrive/telecom_churn.csv"

    X_train, y_train, X_test, y_test, X_train_shape, X_test_shape, X_std_scaler, X_mm_scaler=load_and_preprocess_data(file_path_train)

    # Print shapes
    print("\nDataset Shapes:")
    print(f"Training set: {X_train_shape}")
    print(f"Test set: {X_test_shape}\n")

    # Build the model
    model = build_model((X_train.shape[1], 1))

    # Train the model
    history = train_model(model, X_train, y_train)

    # Track the number of epochs where early stopping occurred
    early_stopped_epoch = len(history.history['loss'])

    # ---> EVALUATE MODEL ON X_test_rfe
    test_loss, test_acc = evaluate_model(model, X_test, y_test)

    # Print results
    final_train_accuracy = history.history['accuracy'][-1]
    print(f"\nFinal Training Accuracy: {final_train_accuracy:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")

run_churn_model()



Dataset Shapes:
Training set: (4568, 15, 1)
Test set: (667, 15, 1)



Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6             │ (None, 15, 1)          │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_42 (Conv1D)        │ (None, 15, 16)         │             64 │ input_layer_6[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_43 (Conv1D)        │ (None, 15, 16)         │             96 │ input_layer_6[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_44 (Conv1D)        │ (None, 15, 16)         │            128 │ input_layer_6[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate_12            │ (None, 15, 48)         │              0 │ conv1d_42[0][0],       │
│ (Concatenate)             │                        │                │ conv1d_43[0][0],       │
│                           │                        │                │ conv1d_44[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_45 (Conv1D)        │ (None, 15, 48)         │             96 │ input_layer_6[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_6 (Add)               │ (None, 15, 48)         │              0 │ concatenate_12[0][0],  │
│                           │                        │                │ conv1d_45[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ average_pooling1d_12      │ (None, 5, 48)          │              0 │ add_6[0][0]            │
│ (AveragePooling1D)        │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_12 (Dropout)      │ (None, 5, 48)          │              0 │ average_pooling1d_12[… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_46 (Conv1D)        │ (None, 5, 32)          │          4,640 │ dropout_12[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_47 (Conv1D)        │ (None, 5, 32)          │          7,712 │ dropout_12[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_48 (Conv1D)        │ (None, 5, 32)          │         10,784 │ dropout_12[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate_13            │ (None, 5, 96)          │              0 │ conv1d_46[0][0],       │
│ (Concatenate)             │                        │                │ conv1d_47[0][0],       │
│                           │                        │                │ conv1d_48[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_49 (Conv1D)        │ (None, 5, 96)          │          4,704 │ dropout_12[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_7 (Add)               │ (None, 5, 96)          │              0 │ concatenate_13[0][0],  │
│                           │                        │                │ conv1d_49[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ average_pooling1d_13 

 Total params: 102,465 (400.25 KB)

 Trainable params: 102,465 (400.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/300
50/50 ━━━━━━━━━━━━━━━━━━━━ 6s 22ms/step - accuracy: 0.6889 - loss: 0.6318 - val_accuracy: 0.0000e+00 - val_loss: 1.1140
Epoch 2/300
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.7180 - loss: 0.5970 - val_accuracy: 0.0000e+00 - val_loss: 1.1325
Epoch 3/300
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.7168 - loss: 0.5880 - val_accuracy: 0.0000e+00 - val_loss: 1.0902
Epoch 4/300
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.7160 - loss: 0.5777 - val_accuracy: 0.2305 - val_loss: 0.8834
Epoch 5/300
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.7150 - loss: 0.5574 - val_accuracy: 0.0117 - val_loss: 1.4412
Epoch 6/300
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.7306 - loss: 0.5400 - val_accuracy: 0.1481 - val_loss: 1.1964
Epoch 7/300
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.7410 - loss: 0.5054 - val_accuracy: 0.2531 - val_loss: 1.1259
Epoch 8/300
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step - accuracy: 0.7670 - loss: 0.4767 - val_

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Concatenate, Input, Dropout, Reshape, LSTM
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from imblearn.over_sampling import SMOTE

def load_and_preprocess_data(file_path):
    df = pd.read_csv(file_path)
    label_encoders = {}

    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
    for col in categorical_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        label_encoders[col] = le

    X = df.drop(columns=["churn"])
    y = df["churn"].values

    #split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # perform feature selection
    rfe = RFE(estimator=RandomForestClassifier(), n_features_to_select=15)
    X_train_rfe = rfe.fit_transform(X_train, y_train)
    X_test_rfe = rfe.transform(X_test)#, y_test)

    # balance classes
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_rfe, y_train)

    #---> Apply StandardScaler to X
    X_standard_scaler = StandardScaler()
    X_train_std = X_standard_scaler.fit_transform(X_train_resampled)
    X_test_std = X_standard_scaler.transform(X_test_rfe)

    # Apply MinMaxScaler after StandardScaler
    # ------> MinMaxScaler IS ALWAYS APPLIED AFTER
    # ------> StandardScaler
    X_min_max_scaler = MinMaxScaler()
    X_train_scaled = X_min_max_scaler.fit_transform(X_train_std)
    X_test_scaled = X_min_max_scaler.transform(X_test_std)

    # Reshape data
    X_train_scaled = X_train_scaled.reshape((X_train_scaled.shape[0], X_train_scaled.shape[1], 1))
    X_test_scaled = X_test_scaled.reshape((X_test_scaled.shape[0], X_test_scaled.shape[1], 1))

    return X_train_scaled, y_train_resampled, X_test_scaled, y_test, X_train_scaled.shape, X_test_scaled.shape, X_standard_scaler, X_min_max_scaler



def build_model(input_shape):
    inputs = Input(shape=input_shape)

    # First Conv1D block
    conv1_block1 = Conv1D(filters=16, kernel_size=3, activation='relu', padding='same')(inputs)
    conv2_block1 = Conv1D(filters=16, kernel_size=5, activation='relu', padding='same')(inputs)
    conv3_block1 = Conv1D(filters=16, kernel_size=7, activation='relu', padding='same')(inputs)

    concat_block1 = Concatenate()([conv1_block1, conv2_block1, conv3_block1])
    pool_block1 = AveragePooling1D(pool_size=3, padding='same')(concat_block1)
    dropout_block1 = Dropout(0.2)(pool_block1)

    # Second Conv1D block
    conv1_block2 = Conv1D(filters=32, kernel_size=3, activation='relu', padding='same')(dropout_block1)
    conv2_block2 = Conv1D(filters=32, kernel_size=5, activation='relu', padding='same')(dropout_block1)
    conv3_block2 = Conv1D(filters=32, kernel_size=7, activation='relu', padding='same')(dropout_block1)

    concat_block2 = Concatenate()([conv1_block2, conv2_block2, conv3_block2])
    pool_block2 = AveragePooling1D(pool_size=3, padding='same')(concat_block2)
    dropout_block2 = Dropout(0.25)(pool_block2)

    # Flatten and reshape before LSTM
    flat = Flatten()(dropout_block2)
    reshaped = Reshape((1, -1))(flat)
    lstm = LSTM(64, return_sequences=False)(reshaped)

    # Fully connected output
    fc = Dense(128, activation='relu')(lstm)
    output = Dense(1, activation='sigmoid')(fc)

    model = Model(inputs=inputs, outputs=output)
    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    model.summary()
    return model


def train_model(model, X_train, y_train):

    history = model.fit(
        X_train, y_train,
        epochs=228,
        batch_size=64,
        verbose=1
    )

    return history

def evaluate_model(model, X_test, y_test):

    test_loss, test_acc = model.evaluate(X_test, y_test)
    return test_loss, test_acc

def run_churn_model():
    file_path_train = "/content/drive/MyDrive/telecom_churn.csv"

    X_train, y_train, X_test, y_test, X_train_shape, X_test_shape, X_standard_scaler, X_min_max_scaler=load_and_preprocess_data(file_path_train)

    # Print shapes
    print("\nDataset Shapes:")
    print(f"Training set: {X_train_shape}")
    print(f"Test set: {X_test_shape}\n")

    # Build the model
    model = build_model((X_train.shape[1], 1))

    # Train the model
    history = train_model(model, X_train, y_train)

    # Track the number of epochs where early stopping occurred
    early_stopped_epoch = len(history.history['loss'])

    # ---> EVALUATE MODEL ON X_test_rfe
    test_loss, test_acc = evaluate_model(model, X_test, y_test)

    # Print results
    final_train_accuracy = history.history['accuracy'][-1]
    print(f"\nFinal Training Accuracy: {final_train_accuracy:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")

run_churn_model()



Dataset Shapes:
Training set: (4568, 15, 1)
Test set: (667, 15, 1)



Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_7             │ (None, 15, 1)          │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_50 (Conv1D)        │ (None, 15, 16)         │             64 │ input_layer_7[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_51 (Conv1D)        │ (None, 15, 16)         │             96 │ input_layer_7[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_52 (Conv1D)        │ (None, 15, 16)         │            128 │ input_layer_7[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate_14            │ (None, 15, 48)         │              0 │ conv1d_50[0][0],       │
│ (Concatenate)             │                        │                │ conv1d_51[0][0],       │
│                           │                        │                │ conv1d_52[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ average_pooling1d_14      │ (None, 5, 48)          │              0 │ concatenate_14[0][0]   │
│ (AveragePooling1D)        │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_14 (Dropout)      │ (None, 5, 48)          │              0 │ average_pooling1d_14[… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_53 (Conv1D)        │ (None, 5, 32)          │          4,640 │ dropout_14[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_54 (Conv1D)        │ (None, 5, 32)          │          7,712 │ dropout_14[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_55 (Conv1D)        │ (None, 5, 32)          │         10,784 │ dropout_14[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate_15            │ (None, 5, 96)          │              0 │ conv1d_53[0][0],       │
│ (Concatenate)             │                        │                │ conv1d_54[0][0],       │
│                           │                        │                │ conv1d_55[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ average_pooling1d_15      │ (None, 2, 96)          │              0 │ concatenate_15[0][0]   │
│ (AveragePooling1D)        │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_15 (Dropout)      │ (None, 2, 96)          │              0 │ average_pooling1d_15[… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ flatten_7 (Flatten)       │ (None, 192)            │              0 │ dropout_15[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ reshape_7 (Reshape)       │ (None, 1, 192)         │              0 │ flatten_7[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ lstm_7 (LSTM)             │ (None, 64)             │         65,792 │ reshape_7[0][0]        │
├──────────────────────

 Total params: 97,665 (381.50 KB)

 Trainable params: 97,665 (381.50 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/228
72/72 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - accuracy: 0.5142 - loss: 0.6919
Epoch 2/228
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.6074 - loss: 0.6589
Epoch 3/228
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.6990 - loss: 0.5940
Epoch 4/228
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.7401 - loss: 0.5349
Epoch 5/228
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7612 - loss: 0.5216
Epoch 6/228
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.7701 - loss: 0.5064
Epoch 7/228
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.7768 - loss: 0.4774
Epoch 8/228
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.7952 - loss: 0.4811
Epoch 9/228
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8040 - loss: 0.4484
Epoch 10/228
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.7932 - loss: 0.4500
Epoch 11/228
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8285 - loss: 0.4130
Epoch 12/228
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accu

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Conv1D, MaxPooling1D, AveragePooling1D, Flatten, Dense, Reshape, LSTM
from tensorflow.keras.layers import Add, Concatenate, Input, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from imblearn.over_sampling import SMOTE

def load_and_preprocess_data(file_path):
    df = pd.read_csv(file_path)
    label_encoders = {}

    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
    for col in categorical_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        label_encoders[col] = le

    X = df.drop(columns=["churn"])
    y = df["churn"].values

    #split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # perform feature selection
    rfe = RFE(estimator=RandomForestClassifier(), n_features_to_select=15)
    X_train_rfe = rfe.fit_transform(X_train, y_train)
    X_test_rfe = rfe.transform(X_test)#, y_test)

    # balance classes
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_rfe, y_train)

    #---> Apply StandardScaler to X
    X_standard_scaler = StandardScaler()
    X_train_std = X_standard_scaler.fit_transform(X_train_resampled)
    X_test_std = X_standard_scaler.transform(X_test_rfe)

    # Apply MinMaxScaler after StandardScaler
    # ------> MinMaxScaler IS ALWAYS APPLIED AFTER
    # ------> StandardScaler
    X_min_max_scaler = MinMaxScaler()
    X_train_scaled = X_min_max_scaler.fit_transform(X_train_std)
    X_test_scaled = X_min_max_scaler.transform(X_test_std)

    # Reshape data
    X_train_scaled = X_train_scaled.reshape((X_train_scaled.shape[0], X_train_scaled.shape[1], 1))
    X_test_scaled = X_test_scaled.reshape((X_test_scaled.shape[0], X_test_scaled.shape[1], 1))

    return X_train_scaled, y_train_resampled, X_test_scaled, y_test, X_train_scaled.shape, X_test_scaled.shape, X_standard_scaler, X_min_max_scaler

def build_model(input_shape):
    inputs = Input(shape=input_shape)

    # First Conv1D block with dilation
    conv1_block1 = Conv1D(filters=16, kernel_size=3, dilation_rate=1, activation='relu', padding='same')(inputs)
    conv2_block1 = Conv1D(filters=16, kernel_size=5, dilation_rate=2, activation='relu', padding='same')(inputs)
    conv3_block1 = Conv1D(filters=16, kernel_size=7, dilation_rate=4, activation='relu', padding='same')(inputs)

    concat_block1 = Concatenate()([conv1_block1, conv2_block1, conv3_block1])
    projection1 = Conv1D(filters=concat_block1.shape[-1], kernel_size=1, padding='same')(inputs)
    skip1 = Add()([concat_block1, projection1])
    pool_block1 = AveragePooling1D(pool_size=3, padding='same')(skip1)
    dropout_block1 = Dropout(0.2)(pool_block1)

    # Second Conv1D block with increased dilation
    conv1_block2 = Conv1D(filters=32, kernel_size=3, dilation_rate=2, activation='relu', padding='same')(dropout_block1)
    conv2_block2 = Conv1D(filters=32, kernel_size=5, dilation_rate=4, activation='relu', padding='same')(dropout_block1)
    conv3_block2 = Conv1D(filters=32, kernel_size=7, dilation_rate=8, activation='relu', padding='same')(dropout_block1)

    concat_block2 = Concatenate()([conv1_block2, conv2_block2, conv3_block2])
    projection2 = Conv1D(filters=concat_block2.shape[-1], kernel_size=1, padding='same')(dropout_block1)
    skip2 = Add()([concat_block2, projection2])
    pool_block2 = AveragePooling1D(pool_size=3, padding='same')(skip2)
    dropout_block2 = Dropout(0.25)(pool_block2)

    # Flatten and reshape before LSTM
    flat = Flatten()(dropout_block2)
    reshaped = Reshape((1, -1))(flat)
    lstm = LSTM(64, return_sequences=False)(reshaped)

    # Fully connected output
    fc = Dense(128, activation='relu')(lstm)
    output = Dense(1, activation='sigmoid')(fc)

    model = Model(inputs=inputs, outputs=output)
    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    model.summary()
    return model




def train_model(model, X_train, y_train):
    early_stopping = EarlyStopping(monitor='val_loss', patience=50, restore_best_weights=True, verbose=1)

    history = model.fit(
        X_train, y_train,
        epochs=300,
        batch_size=64,
        validation_split=0.30,
        callbacks=[early_stopping],
        verbose=1
    )

    return history

def evaluate_model(model, X_test, y_test):

    test_loss, test_acc = model.evaluate(X_test, y_test)
    return test_loss, test_acc

def run_churn_model():
    file_path_train = "/content/drive/MyDrive/telecom_churn.csv"

    X_train, y_train, X_test, y_test, X_train_shape, X_test_shape, X_std_scaler, X_mm_scaler=load_and_preprocess_data(file_path_train)

    # Print shapes
    print("\nDataset Shapes:")
    print(f"Training set: {X_train_shape}")
    print(f"Test set: {X_test_shape}\n")

    # Build the model
    model = build_model((X_train.shape[1], 1))

    # Train the model
    history = train_model(model, X_train, y_train)

    # Track the number of epochs where early stopping occurred
    early_stopped_epoch = len(history.history['loss'])

    # ---> EVALUATE MODEL ON X_test_rfe
    test_loss, test_acc = evaluate_model(model, X_test, y_test)

    # Print results
    final_train_accuracy = history.history['accuracy'][-1]
    print(f"\nFinal Training Accuracy: {final_train_accuracy:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")

run_churn_model()



Dataset Shapes:
Training set: (4568, 15, 1)
Test set: (667, 15, 1)



Model: "functional_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_11            │ (None, 15, 1)          │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_86 (Conv1D)        │ (None, 15, 16)         │             64 │ input_layer_11[0][0]   │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_87 (Conv1D)        │ (None, 15, 16)         │             96 │ input_layer_11[0][0]   │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_88 (Conv1D)        │ (None, 15, 16)         │            128 │ input_layer_11[0][0]   │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate_22            │ (None, 15, 48)         │              0 │ conv1d_86[0][0],       │
│ (Concatenate)             │                        │                │ conv1d_87[0][0],       │
│                           │                        │                │ conv1d_88[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_89 (Conv1D)        │ (None, 15, 48)         │             96 │ input_layer_11[0][0]   │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_20 (Add)              │ (None, 15, 48)         │              0 │ concatenate_22[0][0],  │
│                           │                        │                │ conv1d_89[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ average_pooling1d_22      │ (None, 5, 48)          │              0 │ add_20[0][0]           │
│ (AveragePooling1D)        │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_22 (Dropout)      │ (None, 5, 48)          │              0 │ average_pooling1d_22[… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_90 (Conv1D)        │ (None, 5, 32)          │          4,640 │ dropout_22[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_91 (Conv1D)        │ (None, 5, 32)          │          7,712 │ dropout_22[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_92 (Conv1D)        │ (None, 5, 32)          │         10,784 │ dropout_22[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate_23            │ (None, 5, 96)          │              0 │ conv1d_90[0][0],       │
│ (Concatenate)             │                        │                │ conv1d_91[0][0],       │
│                           │                        │                │ conv1d_92[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_93 (Conv1D)        │ (None, 5, 96)          │          4,704 │ dropout_22[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_21 (Add)              │ (None, 5, 96)          │              0 │ concatenate_23[0][0],  │
│                           │                        │                │ conv1d_93[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ average_pooling1d_23 

 Total params: 102,465 (400.25 KB)

 Trainable params: 102,465 (400.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/300
50/50 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.6975 - loss: 0.6347 - val_accuracy: 0.0000e+00 - val_loss: 1.3048
Epoch 2/300
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7093 - loss: 0.6050 - val_accuracy: 0.0000e+00 - val_loss: 1.3944
Epoch 3/300
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.7140 - loss: 0.5969 - val_accuracy: 0.0000e+00 - val_loss: 1.3005
Epoch 4/300
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7273 - loss: 0.5705 - val_accuracy: 0.0000e+00 - val_loss: 1.4088
Epoch 5/300
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.7040 - loss: 0.5844 - val_accuracy: 0.0751 - val_loss: 1.1518
Epoch 6/300
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7167 - loss: 0.5613 - val_accuracy: 0.1619 - val_loss: 1.0740
Epoch 7/300
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7399 - loss: 0.5233 - val_accuracy: 0.1568 - val_loss: 1.1336
Epoch 8/300
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7202 - loss: 0.5384 - 

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Concatenate, Input, Dropout, Reshape, LSTM
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from imblearn.over_sampling import SMOTE

def load_and_preprocess_data(file_path):
    df = pd.read_csv(file_path)
    label_encoders = {}

    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
    for col in categorical_cols:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        label_encoders[col] = le

    X = df.drop(columns=["churn"])
    y = df["churn"].values

    #split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # perform feature selection
    rfe = RFE(estimator=RandomForestClassifier(), n_features_to_select=15)
    X_train_rfe = rfe.fit_transform(X_train, y_train)
    X_test_rfe = rfe.transform(X_test)#, y_test)

    # balance classes
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_rfe, y_train)

    #---> Apply StandardScaler to X
    X_standard_scaler = StandardScaler()
    X_train_std = X_standard_scaler.fit_transform(X_train_resampled)
    X_test_std = X_standard_scaler.transform(X_test_rfe)

    # Apply MinMaxScaler after StandardScaler
    # ------> MinMaxScaler IS ALWAYS APPLIED AFTER
    # ------> StandardScaler
    X_min_max_scaler = MinMaxScaler()
    X_train_scaled = X_min_max_scaler.fit_transform(X_train_std)
    X_test_scaled = X_min_max_scaler.transform(X_test_std)

    # Reshape data
    X_train_scaled = X_train_scaled.reshape((X_train_scaled.shape[0], X_train_scaled.shape[1], 1))
    X_test_scaled = X_test_scaled.reshape((X_test_scaled.shape[0], X_test_scaled.shape[1], 1))

    return X_train_scaled, y_train_resampled, X_test_scaled, y_test, X_train_scaled.shape, X_test_scaled.shape, X_standard_scaler, X_min_max_scaler



def build_model(input_shape):
    inputs = Input(shape=input_shape)

    # First Conv1D block with dilation
    conv1_block1 = Conv1D(filters=16, kernel_size=3, dilation_rate=1, activation='relu', padding='same')(inputs)
    conv2_block1 = Conv1D(filters=16, kernel_size=5, dilation_rate=2, activation='relu', padding='same')(inputs)
    conv3_block1 = Conv1D(filters=16, kernel_size=7, dilation_rate=4, activation='relu', padding='same')(inputs)

    concat_block1 = Concatenate()([conv1_block1, conv2_block1, conv3_block1])
    projection1 = Conv1D(filters=concat_block1.shape[-1], kernel_size=1, padding='same')(inputs)
    skip1 = Add()([concat_block1, projection1])
    pool_block1 = AveragePooling1D(pool_size=3, padding='same')(skip1)
    dropout_block1 = Dropout(0.2)(pool_block1)

    # Second Conv1D block with increased dilation
    conv1_block2 = Conv1D(filters=32, kernel_size=3, dilation_rate=2, activation='relu', padding='same')(dropout_block1)
    conv2_block2 = Conv1D(filters=32, kernel_size=5, dilation_rate=4, activation='relu', padding='same')(dropout_block1)
    conv3_block2 = Conv1D(filters=32, kernel_size=7, dilation_rate=8, activation='relu', padding='same')(dropout_block1)

    concat_block2 = Concatenate()([conv1_block2, conv2_block2, conv3_block2])
    projection2 = Conv1D(filters=concat_block2.shape[-1], kernel_size=1, padding='same')(dropout_block1)
    skip2 = Add()([concat_block2, projection2])
    pool_block2 = AveragePooling1D(pool_size=3, padding='same')(skip2)
    dropout_block2 = Dropout(0.25)(pool_block2)

    # Flatten and reshape before LSTM
    flat = Flatten()(dropout_block2)
    reshaped = Reshape((1, -1))(flat)
    lstm = LSTM(64, return_sequences=False)(reshaped)

    # Fully connected output
    fc = Dense(128, activation='relu')(lstm)
    output = Dense(1, activation='sigmoid')(fc)

    model = Model(inputs=inputs, outputs=output)
    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    model.summary()
    return model


def train_model(model, X_train, y_train):

    history = model.fit(
        X_train, y_train,
        epochs=183,
        batch_size=64,
        verbose=1
    )

    return history

def evaluate_model(model, X_test, y_test):

    test_loss, test_acc = model.evaluate(X_test, y_test)
    return test_loss, test_acc

def run_churn_model():
    file_path_train = "/content/drive/MyDrive/telecom_churn.csv"

    X_train, y_train, X_test, y_test, X_train_shape, X_test_shape, X_standard_scaler, X_min_max_scaler=load_and_preprocess_data(file_path_train)

    # Print shapes
    print("\nDataset Shapes:")
    print(f"Training set: {X_train_shape}")
    print(f"Test set: {X_test_shape}\n")

    # Build the model
    model = build_model((X_train.shape[1], 1))

    # Train the model
    history = train_model(model, X_train, y_train)

    # Track the number of epochs where early stopping occurred
    early_stopped_epoch = len(history.history['loss'])

    # ---> EVALUATE MODEL ON X_test_rfe
    test_loss, test_acc = evaluate_model(model, X_test, y_test)

    # Print results
    final_train_accuracy = history.history['accuracy'][-1]
    print(f"\nFinal Training Accuracy: {final_train_accuracy:.4f}")
    print(f"Test Accuracy: {test_acc:.4f}")

run_churn_model()



Dataset Shapes:
Training set: (4568, 15, 1)
Test set: (667, 15, 1)



Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2             │ (None, 15, 1)          │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_12 (Conv1D)        │ (None, 15, 16)         │             64 │ input_layer_2[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_13 (Conv1D)        │ (None, 15, 16)         │             96 │ input_layer_2[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_14 (Conv1D)        │ (None, 15, 16)         │            128 │ input_layer_2[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate_4             │ (None, 15, 48)         │              0 │ conv1d_12[0][0],       │
│ (Concatenate)             │                        │                │ conv1d_13[0][0],       │
│                           │                        │                │ conv1d_14[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_15 (Conv1D)        │ (None, 15, 48)         │             96 │ input_layer_2[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add (Add)                 │ (None, 15, 48)         │              0 │ concatenate_4[0][0],   │
│                           │                        │                │ conv1d_15[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ average_pooling1d_4       │ (None, 5, 48)          │              0 │ add[0][0]              │
│ (AveragePooling1D)        │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_4 (Dropout)       │ (None, 5, 48)          │              0 │ average_pooling1d_4[0… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_16 (Conv1D)        │ (None, 5, 32)          │          4,640 │ dropout_4[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_17 (Conv1D)        │ (None, 5, 32)          │          7,712 │ dropout_4[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_18 (Conv1D)        │ (None, 5, 32)          │         10,784 │ dropout_4[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ concatenate_5             │ (None, 5, 96)          │              0 │ conv1d_16[0][0],       │
│ (Concatenate)             │                        │                │ conv1d_17[0][0],       │
│                           │                        │                │ conv1d_18[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv1d_19 (Conv1D)        │ (None, 5, 96)          │          4,704 │ dropout_4[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_1 (Add)               │ (None, 5, 96)          │              0 │ concatenate_5[0][0],   │
│                           │                        │                │ conv1d_19[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ average_pooling1d_5  

 Total params: 102,465 (400.25 KB)

 Trainable params: 102,465 (400.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/183
72/72 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.5365 - loss: 0.6895
Epoch 2/183
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.6100 - loss: 0.6586
Epoch 3/183
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.6757 - loss: 0.6008
Epoch 4/183
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7266 - loss: 0.5524
Epoch 5/183
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.7465 - loss: 0.5240
Epoch 6/183
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.7585 - loss: 0.5097
Epoch 7/183
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7758 - loss: 0.4850
Epoch 8/183
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8003 - loss: 0.4578
Epoch 9/183
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.8092 - loss: 0.4509
Epoch 10/183
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.8004 - loss: 0.4548
Epoch 11/183
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8181 - loss: 0.4337
Epoch 12/183
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - 